In [ ]:
WORKER_CODE = r'''"""Worker (1 GPU): TTT con una demo APARTADA, midiendo dos curvas en paralelo.

Pregunta (del triage, 22-ago): el punto optimo de parada del TTT varia brutalmente por tarea
(5 mejoran hasta 48+, 8 sobreajustan con optimo en 12-36, 7 se danan desde el 12). Para
injertar early stopping en produccion hace falta un criterio OBSERVABLE. Candidato estandar:
leave-one-out -- apartar una demostracion, monitorear su NLL durante el TTT, parar cuando
sube. Este worker mide, por tarea y cada 6 pasos:
  - nll_loo: NLL de la salida de la demo apartada (OBSERVABLE en produccion)
  - nll_gt:  NLL de la verdad del test (ORACULO, solo para correlacionar)
Si los minimos de ambas curvas correlacionan, el early stopping es injertable.

Nota de diseno: se entrena con k-1 demos (una menos que el triage), asi que las curvas no son
identicas a las suyas -- la pregunta es la CORRELACION de optimos, no los valores.
"""
import json
import os
import random
import time

import numpy as np
import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

SHARD = int(os.environ.get("SHARD", "0"))
N_SHARDS = int(os.environ.get("N_SHARDS", "1"))
T00 = time.time()
_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
N_AUG = 12
EPOCAS_TTT = 4
CADA = 6
LR = 5e-5
MAX_SEQ = 3500

import glob
_COMP = None
for c in glob.glob("/kaggle/input/**/arc-agi_evaluation_challenges.json", recursive=True):
    _COMP = os.path.dirname(c)
    break
MDIR = None
for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
    d = os.path.dirname(c)
    if glob.glob(os.path.join(d, "*.safetensors")):
        MDIR = d
        break
assert _COMP and MDIR

CHAL = json.load(open(f"{_COMP}/arc-agi_evaluation_challenges.json"))
SOLS = json.load(open(f"{_COMP}/arc-agi_evaluation_solutions.json"))

KEYS = ["bf45cf4b", "38007db0", "6e453dd6",
        "e8686506", "20270e3b", "7b5033c1", "b0039139", "78332cb0", "28a6681f", "d35bdbdc",
        "dbff022c", "4c3d4a41", "446ef5d2", "dd6b8c4b", "3dc255db", "16de56c4", "97d7923e",
        "2b83f449", "faa9f03d", "eee78d87"]
MIS = [(gi, k) for gi, k in enumerate(KEYS) if gi % N_SHARDS == SHARD]
print(f"[shard {SHARD}] {len(MIS)} tareas: {[k for _, k in MIS]}", flush=True)


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens_mask(inp, out):
    t = [IM_START, USER, NL] + grid_tokens(inp) + [EOS]
    m = [0] * len(t)
    head = [IM_START, ASSIST, NL]
    body = grid_tokens(out) + [EOS]
    return t + head + body, m + [0] * len(head) + [1] * len(body)


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


def d4(g, k):
    a = np.array(g)
    if k >= 4:
        a = a.T
    for _ in range(k % 4):
        a = np.rot90(a)
    return a.tolist()


def aplicar(g, k, perm):
    return [[perm[c] for c in fila] for fila in d4(g, k)]


def build_ttt(pares, seed):
    rng = random.Random(seed)
    seqs = []
    for v in range(N_AUG):
        if v == 0:
            k, perm, orden = 0, list(range(10)), list(range(len(pares)))
        else:
            k = rng.randrange(8)
            perm = list(range(10))
            rng.shuffle(perm)
            orden = list(range(len(pares)))
            rng.shuffle(orden)
        toks, mask = [], []
        for idx in orden:
            i, o = pares[idx]
            t, m = pair_tokens_mask(aplicar(i, k, perm), aplicar(o, k, perm))
            toks += t
            mask += m
        if len(toks) <= MAX_SEQ:
            seqs.append((toks, mask))
    return seqs


base = AutoModelForCausalLM.from_pretrained(MDIR, torch_dtype=torch.bfloat16,
                                            device_map="cuda:0", attn_implementation="sdpa")
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, \
    set_peft_model_state_dict

LCFG = LoraConfig(r=256, lora_alpha=32, use_rslora=True, lora_dropout=0.0, bias="none",
                  task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj",
                                  "embed_tokens", "lm_head"])
MODEL = get_peft_model(base, LCFG)
for _, p_ in MODEL.named_parameters():
    if p_.requires_grad:
        p_.data = p_.data.float()
MODEL.enable_input_require_grads()
MODEL.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
INIT = {k: v.detach().clone() for k, v in get_peft_model_state_dict(MODEL).items()}
try:
    import bitsandbytes as bnb
    OPT_CLS = bnb.optim.AdamW8bit
except Exception:                                            # noqa: BLE001
    OPT_CLS = torch.optim.AdamW
print(f"[shard {SHARD}] modelo listo | mem={torch.cuda.memory_allocated()/1e9:.1f}GB "
      f"({time.time()-T00:.0f}s)", flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


@torch.no_grad()
def nll_total(prompt, ans):
    last, cache = prefill(prompt)
    parts = [last.float()]
    if len(ans) > 1:
        with sdpa_kernel(_SDPA):
            o = MODEL(input_ids=torch.tensor([ans[:-1]], device="cuda:0", dtype=torch.long),
                      past_key_values=cache, use_cache=True, return_dict=True)
        parts.append(o.logits[0].float())
    lg = torch.cat([x.view(-1, x.shape[-1]) for x in parts], 0)
    v = -torch.log_softmax(lg, -1).gather(
        1, torch.tensor(ans, device="cuda:0").unsqueeze(1)).sum().item()
    del parts, lg, cache
    torch.cuda.empty_cache()
    return v


def modo_eval():
    MODEL.eval()
    MODEL.config.use_cache = True


def modo_train():
    MODEL.config.use_cache = False
    MODEL.train()


# gate barato de carga (la tarea que el 4B modela bien)
set_peft_model_state_dict(MODEL, {k: v.clone() for k, v in INIT.items()})
modo_eval()
_t = CHAL["38007db0"]
_p = []
for _x in _t["train"]:
    _tt, _ = pair_tokens_mask(_x["input"], _x["output"])
    _p += _tt
_p += query_tokens(_t["test"][0]["input"])
_n = nll_total(_p, grid_tokens(SOLS["38007db0"][0]) + [EOS])
print(f"[shard {SHARD}] gate 38007db0: {_n:.3f}", flush=True)
assert _n < 1.0, f"gate FALLA ({_n:.3f})"

RES = {"tareas": []}
for gi, k in MIS:
    task = CHAL[k]
    pares = [(p["input"], p["output"]) for p in task["train"]]
    if len(pares) < 2:
        print(f"[{k}] solo {len(pares)} demos, se salta", flush=True)
        continue
    j_out = len(pares) - 1                     # se aparta la ULTIMA demo
    train_pares = pares[:j_out]
    demo_in, demo_out = pares[j_out]

    # prompts de evaluacion (identidad):
    ctx_train = []
    for i_, o_ in train_pares:
        t, _ = pair_tokens_mask(i_, o_)
        ctx_train += t
    p_loo = ctx_train + query_tokens(demo_in)          # contexto k-1 -> demo apartada
    a_loo = grid_tokens(demo_out) + [EOS]
    ctx_full = []
    for i_, o_ in pares:
        t, _ = pair_tokens_mask(i_, o_)
        ctx_full += t
    p_gt = ctx_full + query_tokens(task["test"][0]["input"])   # contexto completo -> test
    a_gt = grid_tokens(SOLS[k][0]) + [EOS]

    set_peft_model_state_dict(MODEL, {kk: v.clone() for kk, v in INIT.items()})
    modo_eval()
    curva = [{"paso": 0, "nll_loo": round(nll_total(p_loo, a_loo), 4),
              "nll_gt": round(nll_total(p_gt, a_gt), 4)}]

    seqs = build_ttt(train_pares, seed=3000 + gi)
    modo_train()
    opt = OPT_CLS([p for p in MODEL.parameters() if p.requires_grad], lr=LR)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=len(seqs) * EPOCAS_TTT)
    paso, losses = 0, []
    t0 = time.time()
    try:
        for ep in range(EPOCAS_TTT):
            for toks, mask in seqs:
                ids = torch.tensor([toks], device="cuda:0", dtype=torch.long)
                lab = torch.tensor([[t if m else -100 for t, m in zip(toks, mask)]],
                                   device="cuda:0", dtype=torch.long)
                with sdpa_kernel(_SDPA):
                    out = MODEL(input_ids=ids, labels=lab)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in MODEL.parameters() if p.requires_grad], 1.0)
                opt.step()
                sch.step()
                opt.zero_grad(set_to_none=True)
                losses.append(out.loss.item())
                paso += 1
                if paso % CADA == 0:
                    MODEL.zero_grad(set_to_none=True)
                    modo_eval()
                    curva.append({"paso": paso,
                                  "nll_loo": round(nll_total(p_loo, a_loo), 4),
                                  "nll_gt": round(nll_total(p_gt, a_gt), 4),
                                  "loss": round(float(np.mean(losses[-CADA:])), 5)})
                    modo_train()
    except torch.cuda.OutOfMemoryError:
        print(f"  [{k}] OOM en paso {paso}", flush=True)
    del opt, sch
    MODEL.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()

    RES["tareas"].append({"task": k, "n_demos": len(pares), "L_loo": len(a_loo),
                          "L_gt": len(a_gt), "curva": curva,
                          "secs": round(time.time() - t0, 1)})
    ml = min(curva, key=lambda c: c["nll_loo"])
    mg = min(curva, key=lambda c: c["nll_gt"])
    print(f"[shard {SHARD}] {k}: optimo LOO paso {ml['paso']} | optimo GT paso {mg['paso']} "
          f"| {time.time()-t0:.0f}s", flush=True)
    json.dump(RES, open(f"/kaggle/working/shard{SHARD}.json", "w"))

json.dump(RES, open(f"/kaggle/working/shard{SHARD}.json", "w"))
print(f"[shard {SHARD}] fin {time.time()-T00:.0f}s", flush=True)
'''

import json
import os
import subprocess
import sys
import time

open("/kaggle/working/worker.py", "w").write(WORKER_CODE)
N_GPUS = 4
T0 = time.time()
procs = []
for i in range(N_GPUS):
    env = dict(os.environ, SHARD=str(i), N_SHARDS=str(N_GPUS),
               CUDA_VISIBLE_DEVICES=str(i), PYTHONUNBUFFERED="1")
    p = subprocess.Popen([sys.executable, "/kaggle/working/worker.py"],
                         stdout=open(f"/kaggle/working/shard{i}.log", "w"),
                         stderr=subprocess.STDOUT, env=env)
    procs.append(p)
    print(f"[main] worker {i} pid {p.pid}", flush=True)
for i, p in enumerate(procs):
    rc = p.wait()
    print(f"[main] worker {i} rc={rc} ({time.time()-T0:.0f}s)", flush=True)
    if rc != 0:
        print(open(f"/kaggle/working/shard{i}.log", errors="replace").read()[-3000:],
              flush=True)
tareas = []
for i in range(N_GPUS):
    f = f"/kaggle/working/shard{i}.json"
    if os.path.exists(f):
        tareas += json.load(open(f))["tareas"]
json.dump({"tareas": tareas}, open("/kaggle/working/ttt_loo.json", "w"))
coinc = 0
for t in tareas:
    ml = min(t["curva"], key=lambda c: c["nll_loo"])["paso"]
    mg = min(t["curva"], key=lambda c: c["nll_gt"])["paso"]
    if abs(ml - mg) <= 6:
        coinc += 1
print("RESULT_JSON:" + json.dumps({"n": len(tareas), "optimos_coinciden_pm6": coinc,
                                   "wall_s": round(time.time() - T0, 1)}), flush=True)
